In [21]:
import ROOT
import math
# import pandas as pd

# print(f"ROOT version: {ROOT.__version__}")
import numpy as np
from scipy.special import j0
import plotly.graph_objects as go
from scipy.integrate import fixed_quad


In [22]:
def read_data_file(filename):
    """Read data file and return arrays for x, y, y_error"""
    x_vals = []
    y_vals = []
    y_errs = []
    
    with open(filename, 'r') as f:
        for line in f:
            if line.strip() and not line.startswith('#'):
                parts = line.split()
                if len(parts) >= 3:
                    x_vals.append(float(parts[0]))
                    y_vals.append(float(parts[1]))
                    y_errs.append(float(parts[2]))
    
    return x_vals, y_vals, y_errs

In [23]:
# Load experimental data for all energies from ATLAS
x_atlas_all, y_atlas_all, yerr_atlas_all = read_data_file('../../../data/ens_atlas_difc0_2.dat')

# Function to process data for each energy block
def process_data(x_data, y_data, yerr_data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        if end is None:
            end = len(x_data)
        x_values.append(x_data[start:end])
        y_values.append(y_data[start:end])
        y_errors.append(yerr_data[start:end])
    
    return x_values, y_values, y_errors


In [24]:
#ranges for each energy 
atlas_blocks = [(0, 29), (29, 58), (58, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(x_atlas_all, y_atlas_all, yerr_atlas_all, atlas_blocks)

# Extract values by energy
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

In [25]:
# defining parameters/constants
b_0 = (33 - 6) / (12 * np.pi)
lambda_qcd = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25


#ensemble parameters
param_mg_atlas_pl = 0.421
param_eps_atlas_pl = 0.0753
param_a1_atlas_pl = 1.517
param_a2_atlas_pl = 2.05

In [26]:
#--------------------------------------
# Eq 22 - GE
#--------------------------------------
def m2_pl(q2, mg):
    lambda2 = lambda_qcd ** 2
    rho_mg_2 = rho * (mg ** 2)
    ratio = math.log((q2 + rho_mg_2) / lambda2) / math.log(rho_mg_2 / lambda2)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

#--------------------------------------
# Eq 26 - GE
#--------------------------------------
def G_p(q2, a1, a2):
    t = -q2
    return np.exp(-(a1 * np.abs(t) + a2 * np.abs(t) ** 2))


#--------------------------------------
# Eq 24 - GE
#--------------------------------------
def alpha_D(q2, mg, m2_type):
    m2_func = m2_type(q2, mg)
    return 1.0 / (b_0 * (q2 + m2_func) * math.log((q2 + 4 * m2_func) / (lambda_qcd ** 2)))

#--------------------------------------
# Eq 7 - GE
#--------------------------------------
def T_1(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    G0 = G_p(q, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2


#--------------------------------------
# Eq 8 - GE
#--------------------------------------
def T_2(k, phi, mg, a1, a2, m2_type, q):
    q2 = q ** 2
    qk_cos = q * k * math.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_type)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_type)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


#--------------------------------------
# Eq 11 - GE
#--------------------------------------
def born_sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

#--------------------------------------
# Eq 6 - GE
#--------------------------------------
def born_amp(diff_T, s, eps, t):
    alpha_pomeron = 1.0 + eps + 0.25 * t
    s_tilde = s/s0

    return 1j * s * 8 * (s_tilde**(alpha_pomeron - 1)) * diff_T

In [27]:
def root_1d_integrator(func, lower_limit, upper_limit):
    """Integrates the given function using ROOT's adaptive integration method.
    args:
        func: The function to be integrated.
        lower_limit: Lower limit of integration.
        upper_limit: Upper limit of integration.
    returns:
        A tuple containing the estimated value of the integral and its uncertainty.
    """
    # creating Functor to be used by ROOT's integrator
    functor = ROOT.Math.Functor1D(func)

    type = ROOT.Math.IntegrationOneDim.kADAPTIVE   # integration type
    absTol = 1e-4
    relTol = 1e-4
    size   = 20
    rule   = ROOT.Math.Integration.kGAUSS15   
        
    # defining Integrator object
    integrator = ROOT.Math.IntegratorOneDim(type, absTol, relTol, size, rule)
    integrator.SetFunction(functor)
    
    # calculating integral and error 
    result = integrator.Integral(lower_limit, upper_limit)
    error = integrator.Error()
    
    return result, error 



In [28]:
def k_integral(k, mg, a1, a2, m2_func, q):
    """"Calculates the integral over k"""
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    return integrand

def phi_integral(phi, mg, a1, a2, m2_func, q, k_max):
    """"Calculates the integral over phi nested in k"""
    def inner_in_k(k):
        return k * (T_1(k, phi, mg, a1, a2, m2_func, q) -
                    T_2(k, phi, mg, a1, a2, m2_func, q))
    
    result, _ = root_1d_integrator(inner_in_k, 0, k_max)
    return result

def compute_k_phi_integral(mg, a1, a2, m2_func, q, k_max):
    """Computes the nested integral over k and phi."""
    result, error = root_1d_integrator(
        lambda phi: phi_integral(phi, mg, a1, a2, m2_func, q, k_max), 0, 2*math.pi)
    return result, error


# TESTING 

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0
k_max = 13000

compute_k_phi_integral(mg, a1, a2, m2_pl, q, k_max)

(7.965987352559131, 8.844022572683941e-14)

In [29]:
# COMPUTING SIGMA TOT BORN 

#start parameters
start_sqrt_s = 100
max_sqrt_s = 13000
step_size = 200

mg = param_mg_atlas_pl
a1 = param_a1_atlas_pl
a2 = param_a2_atlas_pl
q = 0

lst_sigma_tot_born_list = []
lst_sqrt_s_list = []

# start initial sqrt for while loop  
current_sqrt_s = start_sqrt_s

while current_sqrt_s <= max_sqrt_s:

    current_s = current_sqrt_s**2

    # calculates diff t (eq 7 and 8)
    diff_t,_ = compute_k_phi_integral(mg, a1, a2, m2_pl, q, current_sqrt_s)

    # calculating born amplitude
    born_amp_value = born_amp(diff_t, current_s, param_eps_atlas_pl, 0)
    # print(born_amp_value)

    #calculating sigma tot born
    born_sigma_tot_value = born_sigma_tot(born_amp_value, current_s)
    # print(born_sigma_tot_value)

    # append results to list
    lst_sigma_tot_born_list.append(born_sigma_tot_value)
    lst_sqrt_s_list.append(current_sqrt_s)

    # increase step
    current_sqrt_s += step_size 
    print(born_sigma_tot_value)

49.64805917299905
58.581020011925
63.26556087305723
66.55401305502859
69.12122172394115
71.24201962839247
73.05708730869492
74.648627715497
76.06906749987411
77.35400020360356
78.52875475197625
79.61202984815023
80.61804280094172
81.55786867688803
82.44031140889837
83.27249099301083
84.0602513953254
84.80845131485145
85.5211761175423
86.20189536702067
86.85358195103674
87.47880355967614
88.07979390330043
88.65850884713969
89.21667115528243
89.75580652041585
90.27727284768216
90.78228425975604
91.27193092991801
91.7471955875209
92.20896734691338
92.65805336649257
93.0951887355824
93.52104489934493
93.93623689733539
94.34132956716718
94.73684293430956
95.1232568836761
95.50101523531839
95.87052931215156
96.2321810741792
96.58632588119833
96.93329493583028
97.2733974504573
97.60692257485076
97.93414111566176
98.25530707429412
98.57065902580314
98.88042135823034
99.18480538906432
99.48401037322813
99.77822441505872
100.06762529509606
100.35233738091726
100.63265151152004
100.90858721859351

In [30]:
def chi_eikonal(s, b, eps, mg, a1, a2, m2_func, born_amp_func):
    """
    Eikonal function χ(s,b) from eq. (23)
    χ(s,b) = (1/s) ∫ q dq J₀(bq) A_Born(s,t)
    where t = -q²
    
    Parameters:
    - s: Mandelstam s (squared center-of-mass energy)
    - b: Impact parameter
    - eps, mg, a1, a2, m2_func: Model parameters  
    - born_amp_func: Function that computes Born amplitude: born_amp(diff_T, s, eps, t)
    """
    def integrand_real(q):
        t = -q**2  
        
        # Calculate diff_T for this q value        
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)
        )

        # Calculate Born amplitude using the provided function
        amp_born = born_amp_func(diff_T, s, eps, t)
        
        # Integrand: q * J₀(b*q) * A_Born(s,t)
        return q * j0(b * q) * amp_born.real
    
    def integrand_imag(q):
        t = -q**2
        
        # Calculate diff_T for this q value        
        diff_T, _ = compute_k_phi_integral(
            mg=mg,
            a1=a1,
            a2=a2,
            m2_func=m2_func,
            q=q,
            k_max=np.sqrt(s)
        )

        # Calculate Born amplitude using the provided function
        amp_born = born_amp_func(diff_T, s, eps, t)
        
        # Integrand: q * J₀(b*q) * A_Born(s,t)
        return q * j0(b * q) * amp_born.imag
    
    real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, 0.2)  
    imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, 0.2)  

    integral_result = real_integral_result + 1j * imag_integral_result

    return integral_result / s

# print(chi_eikonal(7000**2, 10.0, param_eps_atlas_pl, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl, born_amp))

In [31]:
#--------------------------------------
# Eq 24 - EIK (using the new chi_eikonal)
#--------------------------------------


def eik_amp(s, t, eps, mg, a1, a2, m2_func, born_amp_func, q_max=0.2):
    """
    Eikonalized amplitude from eq. (24)
    A_eik(s,t) = i s ∫ b db J₀(b√(-t)) [1 - exp(iχ(s,b))]
    where χ(s,b) is computed using the new chi_eikonal function
    
    Parameters:
    - s: Mandelstam s (squared center-of-mass energy)
    - t: Mandelstam t (momentum transfer squared, negative)
    - eps, mg, a1, a2, m2_func: Model parameters
    - born_amp_func: Function that computes Born amplitude: born_amp(diff_T, s, eps, t)
    - q_max: Maximum q for χ integration
    """
    q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
    def integrand_real(b_val):
        # Calculate eikonal function χ(s,b) using the new chi_eikonal
        chi_val = chi_eikonal(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func)
        
        # Compute [1 - exp(iχ(s,b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
        return (b_val * j0(b_val * q) * one_minus_exp).real
    
    def integrand_imag(b_val):
        # Calculate eikonal function χ(s,b) using the new chi_eikonal
        chi_val = chi_eikonal(s, b_val, eps, mg, a1, a2, m2_func, born_amp_func)
        
        # Compute [1 - exp(iχ(s,b))]
        exp_term = np.exp(1j * chi_val)
        one_minus_exp = 1.0 - exp_term
        
        # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
        return (b_val * j0(b_val * q) * one_minus_exp).imag
    
    # Integrate real and imaginary parts separately
    real_integral, _ = root_1d_integrator(integrand_real, 0.0, 30)
    imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, 30)
    
    integral_result = real_integral + 1j * imag_integral
    
    # Final amplitude: i s times the integral
    return 1j * s * integral_result

# print(eik_amp(7000**2,-0.04,param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl, born_amp))

In [ ]:
import ROOT
import numpy as np
import math


def root_minimize(func,
                ndim,
                minimizerName="Minuit2",
                algoName="",
                mg_init=None,
                eps_init=None,
                a1_init=None,
                a2_init=None,
                stepSize=None,
                maxFunctionCalls=1000000,
                maxIterations=10000,
                tolerance=1e-8,
                printLevel=1):
    """
    Generic ROOT Minimization Wrapper with Confidence Level Calculation.

    Parameters
    ----------
    func : callable
        Function to minimize. Should accept a list or numpy array of length `ndim`.

    ndim : int
        Number of parameters to minimize.

    minimizerName : str, default="Minuit2"
        Minimizer to use (Minuit, Minuit2, GSLMultiMin, GSLSimAn, Genetic, etc.).
    
    algoName : str, default=""
        Specific algorithm (Migrad, BFGS, ConjugateFR, Simplex, etc.).
    
    mg_init : float or None
        Initial guess for parameter 'mg'. Defaults to 0.0 if None.
    
    eps_init : float or None
        Initial guess for parameter 'eps'. Defaults to 0.0 if None.
    
    a1_init : float or None
        Initial guess for parameter 'a1'. Defaults to 0.0 if None.
    
    a2_init : float or None
        Initial guess for parameter 'a2'. Defaults to 0.0 if None.
    
    stepSize : list of floats or None
        Step sizes for each parameter. Defaults to 0.01 for all.
    
    maxFunctionCalls : int, default=1000000
        Maximum allowed function evaluations.
    
    maxIterations : int, default=10000
        Maximum allowed iterations.
    
    tolerance : float, default=1e-8
        Desired tolerance for convergence.
    
    printLevel : int, default=1
        Verbosity of the minimizer (0=quiet, 1=normal, 2=verbose).

    Returns
    -------
    dict
        Dictionary containing:
        - 'success': bool, whether minimization converged successfully
        - 'x': numpy array, parameter values at minimum
        - 'status': int, minimizer status (0 = success)
        - 'hesse_errors': numpy array, symmetric Hesse errors
        - 'minos_errors_low': numpy array, lower MINOS errors
        - 'minos_errors_up': numpy array, upper MINOS errors
    """

    #-------------------
    #  SET STARTING POINT
    #-------------------

    param_names = ["mg", "eps", "a1", "a2"]
    init_map = [mg_init, eps_init, a1_init, a2_init]
    startPoint = []
    for i in range(ndim):
        if init_map[i] is not None:
            startPoint.append(init_map[i])
        else:
            startPoint.append(0.0)  # fallback default
    # --------------------------------------------------------------

    #-------------------
    #  SET STEP SIZE
    #-------------------
    if stepSize is None:
        stepSize = [0.01] * ndim

    #-------------------
    #  SET CONFIDENCE LEVEL FOR 4D CASE 90% CL
    #-------------------
    errordef = 7.78

    # import scipy
    # import scipy.stats
    # print(scipy.stats.chi2.ppf(0.9 , df=4))

    # cl_to_errordef_4d = {
    #     68.3: 4.72,
    #     90.0: 7.78,
    #     95.0: 9.49,
    #     99.0: 13.28
    # }

    
    #-------------------
    #  CREATE MINIMIZER
    #-------------------

    minimizer = ROOT.Math.Factory.CreateMinimizer(minimizerName, algoName)
    if not minimizer:
        raise RuntimeError(f"Cannot create minimizer \"{minimizerName}\"")
    

    #-------------------
    #  SET OPTIONS
    #-------------------

    minimizer.SetMaxFunctionCalls(maxFunctionCalls)
    minimizer.SetMaxIterations(maxIterations)
    minimizer.SetTolerance(tolerance)
    minimizer.SetPrintLevel(printLevel)
    minimizer.SetErrorDef(errordef)
    f = ROOT.Math.Functor(func, ndim)
    minimizer.SetFunction(f)

    
    variable = list(startPoint)

    #-------------------
    #  SET PARAMETERS
    #-------------------

    # renaming variable names to match model parameters and set parameters 
    for i in range(ndim):
        if i < len(param_names):
            name = param_names[i]
        else:
            name = f"x{i}"
        minimizer.SetVariable(i, name, variable[i], stepSize[i])

    #could replace the code above by the following code to set parameters without renaming

    # for i in range(ndim):
    #     minimizer.SetVariable(i, f"x{i}", variable[i], stepSize[i])


    #-------------------
    #  RUN MINIMIZATION
    #-------------------

    minimization = minimizer.Minimize()
    if not minimization:
        return {'success': False}
    

    #-------------------
    # GET HESSE ERROR
    #-------------------

    # Create empty arrays to store the results
    xs = np.zeros(ndim)           # parameter values at minimum
    hesse_errors = np.zeros(ndim) # symmetric Hesse errors

    # Loop over each parameter and extract the value and Hesse error
    for i in range(ndim):
        xs[i] = minimizer.X()[i]          # get the fitted value of parameter i
        hesse_errors[i] = minimizer.Errors()[i]  # get the Hesse error for parameter i


    #-------------------
    # GET MINOS ERROR
    #-------------------
    
    # Initialize arrays to store MINOS errors
    minos_errors_low = np.zeros(ndim)
    minos_errors_up = np.zeros(ndim)

    # Temporary arrays for ROOT's GetMinosError
    errLow = np.zeros(1, dtype=np.float64)
    errUp  = np.zeros(1, dtype=np.float64)

    for i in range(ndim):
        success = minimizer.GetMinosError(i, errLow, errUp)
        if success:
            minos_errors_low[i] = errLow[0]
            minos_errors_up[i] = errUp[0]
        else:
            # fallback to Hesse errors if MINOS fails
            minos_errors_low[i] = -hesse_errors[i]
            minos_errors_up[i] = hesse_errors[i]



    # print results
    print("\nMinimization results (values ± Hesse ± MINOS):")
    for i in range(ndim):
        print(f"{param_names[i]}: {xs[i]:.6f} "
              f"± {hesse_errors[i]:.6f} "
              f"[{minos_errors_low[i]:+.6f}, {minos_errors_up[i]:+.6f}]")

    print(f"\nStatus: {minimizer.Status()} (0 = success)\n")
    # ----------------------

    return {
        'success': minimization and minimizer.Status() == 0,
        'x': xs,
        'status': minimizer.Status(),
        'hesse_errors': hesse_errors,
        'minos_errors_low': minos_errors_low,
        'minos_errors_up': minos_errors_up,
    }


In [33]:
# #--------------------------------------
# # Eq 23 - EIK
# #--------------------------------------

# q_max = 0.2

# def chi_eikonal(s, b, eps, mg, a1, a2, m2_func):
#     """
#     Eikonal function χ(s,b) from eq. (23)
#     χ(s,b) = (1/s) ∫ q dq J₀(bq) A_Born(s,t)
#     where t = -q²
#     """
#     def integrand_real(q):
#         t = -q**2  
        
#         # Calculate diff_T for this q value        
#         diff_T, _ = compute_k_phi_integral(
#             mg=mg,
#             a1=a1,
#             a2=a2,
#             m2_func=m2_func,
#             q=q,
#             k_max=np.sqrt(s)  # Use sqrt(s) as k_max
#         )

#         # Calculate Born amplitude for this t
#         amp_born = born_amp(diff_T, s, eps, t)
        
#         # Integrand: q * J₀(b*q) * A_Born(s,t)
#         return q * j0(b * q) * amp_born.real
    
#     def integrand_imag(q):
#         t = -q**2  # t = -q² as specified
        
#         # Calculate diff_T for this q value        
#         diff_T, _ = compute_k_phi_integral(
#             mg=mg,
#             a1=a1,
#             a2=a2,
#             m2_func=m2_func,
#             q=q,
#             k_max=np.sqrt(s)  # Use sqrt(s) as k_max
#         )

#         # Calculate Born amplitude for this t
#         amp_born = born_amp(diff_T, s, eps, t)
        
#         # Integrand: q * J₀(b*q) * A_Born(s,t)
#         return q * j0(b * q) * amp_born.imag
    
#     real_integral_result, _ = root_1d_integrator(integrand_real, 0.0, q_max)  
#     imag_integral_result, _ = root_1d_integrator(integrand_imag, 0.0, q_max)  

#     integral_result = real_integral_result + 1j*imag_integral_result

#     return integral_result / s

# print(chi_eikonal(7000**2, 10.0, param_eps_atlas_pl, param_mg_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl,m2_pl))

# #--------------------------------------
# # Eq 24 - EIK
# #--------------------------------------

# b_max = 30  # Maximum impact parameter

# def eikonal_amplitude(s, t, eps, mg, a1, a2, m2_func):
#     """
#     Eikonalized amplitude from eq. (24)
#     A_eik(s,t) = i s ∫ b db J₀(b√(-t)) [1 - exp(iχ(s,b))]
#     where t is the Mandelstam variable (negative)
#     """
#     q = np.sqrt(-t)  # q = √(-t) since t = -q²
    
#     def integrand_real(b):
#         # Calculate eikonal function χ(s,b)
#         chi_val = chi_eikonal(s, b, eps, mg, a1, a2, m2_func)
        
#         # Compute [1 - exp(iχ(s,b))]
#         exp_term = np.exp(1j * chi_val)
#         one_minus_exp = 1.0 - exp_term
        
#         # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
#         return (b * j0(b * 0) * one_minus_exp).real
    
#     def integrand_imag(b):
#         # Calculate eikonal function χ(s,b)
#         chi_val = chi_eikonal(s, b, eps, mg, a1, a2, m2_func)
        
#         # Compute [1 - exp(iχ(s,b))]
#         exp_term = np.exp(1j * chi_val)
#         one_minus_exp = 1.0 - exp_term
        
#         # Integrand: b * J₀(b√(-t)) * [1 - exp(iχ(s,b))]
#         return (b * j0(b * 0) * one_minus_exp).imag
    
#     # Integrate real and imaginary parts separately
#     real_integral, _ = root_1d_integrator(integrand_real, 0.0, b_max)
#     imag_integral, _ = root_1d_integrator(integrand_imag, 0.0, b_max)
    
#     integral_result = real_integral + 1j * imag_integral
    
#     # Final amplitude: i s times the integral
#     return 1j * s * integral_result

# print(eikonal_amplitude(7000**2,-0.04,param_eps_atlas_pl,param_mg_atlas_pl,param_a1_atlas_pl,param_a2_atlas_pl,m2_pl))

In [ ]:


import ROOT
import numpy as np
import math
import array


def chi2_dof(func, x_data, y_data, y_errors, params_init, param_names=None, 
             minimizer_options=None, confidence_level=90.0):
    """
    Calculate chi2/dof for a given function and data using GenericMinimization.
    
    Parameters:
    -----------
    func : callable
        Function of the form func(params) where params is a list/array of parameters
    x_data : list or array
        x-values of the data points
    y_data : list or array 
        y-values of the data points
    y_errors : list or array
        Uncertainties on y-data points
    params_init : list
        Initial values for the parameters
    param_names : list of str, optional
        Names for parameters (default: ["mg", "eps", "a1", "a2", ...])
    param_limits : list of tuples, optional
        Limits for each parameter as [(min1, max1), (min2, max2), ...]
    param_fixed : list of bool, optional
        Which parameters to fix [False, True, ...]
    minimizer_options : dict, optional
        Options for minimizer
    confidence_level : float, default=90.0
        Confidence level for error calculation
        
    Returns:
    --------
    dict : Dictionary containing chi2, ndf, chi2_dof, and fit results
    """
    
    ndim = len(params_init)
    n_data = len(x_data)
    
    # Convert data to arrays for efficient access
    # Flatten nested lists and convert to arrays
    x_arr = np.array([xi for sublist in x_data for xi in sublist], dtype=float)
    y_arr = np.array([yi for sublist in y_data for yi in sublist], dtype=float)
    err_arr = np.array([ei for sublist in y_errors for ei in sublist], dtype=float)

    
    def chi2_function(params):
        chi2_val = 0.0
        for i in range(n_data):
            # Calculate model prediction at x_arr[i] using the parameters
            model_val = func(x_arr[i], params)
            diff = (y_arr[i] - model_val) / err_arr[i]
            chi2_val += diff * diff
        return chi2_val
    
 
    mg_init, eps_init, a1_init, a2_init = None, None, None, None
    if ndim >= 1:
        mg_init = params_init[0]
    if ndim >= 2:
        eps_init = params_init[1]
    if ndim >= 3:
        a1_init = params_init[2]
    if ndim >= 4:
        a2_init = params_init[3]
    
    # Set up minimizer options
    if minimizer_options is None:
        minimizer_options = {}
    
    # Run minimization
    result = root_minimize(
        func=chi2_function,
        ndim=ndim,
        mg_init=mg_init,
        eps_init=eps_init,
        a1_init=a1_init,
        a2_init=a2_init
    )
    
    # Calculate degrees of freedom
    ndf = n_data - ndim
    
    # Get chi2 value at minimum
    chi2_min = chi2_function(result['x'])
    chi2_dof_val = chi2_min / ndf if ndf > 0 else float('nan')
    
    # Prepare output
    output = {
        'chi2': chi2_min,
        'ndf': ndf,
        'chi2_dof': chi2_dof_val,
        'success': result['success'],
        'fitted_params': result['x'],
        'hesse_errors': result['hesse_errors'],
        'minos_errors_low': result['minos_errors_low'],
        'minos_errors_up': result['minos_errors_up'],
        'status': result['status']
    }
    
    # Print results
    print("\n" + "="*60)
    print("CHI2/DOF FIT RESULTS")
    print("="*60)
    print(f"Chi2        = {chi2_min:.8f}")
    print(f"NDF         = {ndf}")
    print(f"Chi2/NDF    = {chi2_dof_val:.8f}")
    print(f"Success     = {result['success']}")
    print(f"Status      = {result['status']} (0 = success)")
    
    if param_names is None:
        param_names = [f"p{i}" for i in range(ndim)]
    
    print("\nFitted parameters:")
    for i in range(ndim):
        print(f"  {param_names[i]:8s}: {result['x'][i]:.6f} ± {result['hesse_errors'][i]:.6f} "
              f"[{result['minos_errors_low'][i]:+.6f}, {result['minos_errors_up'][i]:+.6f}]")
    print("="*60)
    
    return output


In [ ]:

# Example usage with the same linear model
if __name__ == "__main__":
    # ---------------------------
    # 3. Calculate chi2/dof using the new function
    # ---------------------------
    result = chi2_dof(
    func=lambda x, p: eik_amp(7000**2, -0.04, p[1], p[0], p[2], p[3], m2_pl, born_amp),
    x_data=x_atlas,
    y_data=y_atlas,
    y_errors=yerr_atlas,
    params_init=[param_mg_atlas_pl, param_eps_atlas_pl, param_a1_atlas_pl, param_a2_atlas_pl],
    minimizer_options={
        'tolerance': 1e-8,
        'maxIterations': 100000,
        'printLevel': 1
    },
    confidence_level=90.0
)
